# 02 — Text Models: RF, SVM, XGBoost (+ optional BERT/RoBERTa)


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

import pandas as pd
from sklearn.model_selection import train_test_split
from src.data.loaders import load_text_data
from src.data.preprocessing import build_text_feature_matrix, scale_features
from src.models.classical_ml import train_and_evaluate
from src.utils import set_seed, ensure_dir

set_seed(42)
SAMPLE = 3000
RUN_TRANSFORMERS = os.environ.get("RUN_TRANSFORMERS", "0") == "1"


In [ ]:
# Classical ML
df = load_text_data(sample=SAMPLE)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["wellbeing_decline"], random_state=42)
X_train, X_test, _, _, _ = build_text_feature_matrix(train_df["text"], test_df["text"])
X_train, X_test, _ = scale_features(X_train, X_test)
y_train, y_test = train_df["wellbeing_decline"].values, test_df["wellbeing_decline"].values

classical_results = []
for model_name in ["random_forest", "svm", "xgboost"]:
    r = train_and_evaluate(model_name, X_train, y_train, X_test, y_test)
    classical_results.append({"model": model_name, **r["test"]})
    print(f"{model_name}: {r['test']}")


In [ ]:
# Transformers (set RUN_TRANSFORMERS=1 to enable — slow on CPU)
transformer_results = []
if RUN_TRANSFORMERS:
    from src.models.text_transformers import train_transformer
    tr_train, tr_val = train_test_split(train_df, test_size=0.15, stratify=train_df["wellbeing_decline"], random_state=42)
    for model_name in ["bert-base-uncased", "roberta-base", "distilbert-base-uncased"]:
        out_dir = ensure_dir(ROOT / "outputs" / "models" / model_name.replace("/", "_"))
        print(f"Training {model_name}...")
        result = train_transformer(
            model_name=model_name,
            train_texts=tr_train["text"].tolist(), train_labels=tr_train["wellbeing_decline"].tolist(),
            val_texts=tr_val["text"].tolist(), val_labels=tr_val["wellbeing_decline"].tolist(),
            output_dir=str(out_dir), num_labels=2, epochs=2, batch_size=8,
        )
        # strip the "eval_" prefix so these rows share accuracy/f1_macro/f1_weighted
        # column names with the classical rows above instead of creating NaN-filled
        # duplicate columns when concatenated into one CSV
        transformer_results.append({
            "model": model_name,
            **{k.replace("eval_", ""): v for k, v in result["eval"].items() if "f1" in k or "accuracy" in k},
        })
        print(result["eval"])
else:
    print("Skipping BERT/RoBERTa (set RUN_TRANSFORMERS=1 to enable)")


In [ ]:
from IPython.display import display
all_text = pd.DataFrame(classical_results + transformer_results)
out = ensure_dir(ROOT / "outputs" / "results")
all_text.to_csv(out / "text_model_results.csv", index=False)
display(all_text)
print("02_text_models.py complete.")
